# Simple setup: airport wastewater & ICU surveillance

This notebook is a **user-facing entry point** for running the airport wastewater + ICU surveillance workflow using **your own**:

- Epidemiological parameters (e.g. $R_0$, generation time, latent & infectious periods)
- Mobility / flight matrix and population data

The notebook helps you:

1. Configure paths to your mobility and population datasets.
2. Run basic preprocessing using the `DataSetup` class from `data_processing.py`.
3. (Optionally) adjust high-level epidemiological parameters.
4. Export a `daily_imports_sensitivity.csv` file in the expected format.
5. Trigger the Julia ICU + wastewater simulations from `NBPMscape/AWW_ICU_examples/full_ICU_WW.jl`.

> **Note**: This notebook does not hide model details; it is meant to be a transparent scaffold that you can adapt to your own scenario.


In [ ]:
from pathlib import Path

import pandas as pd

# Paths relative to this notebook
PROJECT_ROOT = Path(__file__).resolve().parents[1]
PGFGLEAM_DIR = PROJECT_ROOT / "pgfgleam"
NBPMSCAPE_DIR = PROJECT_ROOT / "NBPMscape"

# Default locations used by the ICU + WW Julia script
DEFAULT_DAILY_IMPORTS = PGFGLEAM_DIR / "daily_imports_sensitivity.csv"
DEFAULT_RESULTS = PGFGLEAM_DIR / "datasets" / "full_result.csv"

print("Project root:", PROJECT_ROOT)
print("pgfgleam dir:", PGFGLEAM_DIR)
print("NBPMscape dir:", NBPMSCAPE_DIR)
print("Default daily imports CSV:", DEFAULT_DAILY_IMPORTS)
print("Default results CSV:", DEFAULT_RESULTS)


## 1. Configure your mobility and population data

Below you can either:

- **Use the existing example data** (recommended for a first run), or
- **Point to your own mobility / population / country-code files**.

The `DataSetup` class in `data_processing.py` expects three CSV files:

- A **flight / flow matrix** (rows = source, columns = destination, cell = monthly passengers or flows)
- A **population file** with at least `Location` and `Population` columns
- A **country-codes file** with at least `country` and `country_code` columns

You can adapt the code cell below to your own file structure.


In [ ]:
from data_processing import DataSetup

# Option A: use example files shipped with pgfgleam (edit these if you move things)
example_flight_matrix = PGFGLEAM_DIR / "pgfgleam" / "datasets" / "country_flow_202001 1.csv"
example_population = PGFGLEAM_DIR / "pgfgleam" / "datasets" / "WPP2019_TotalPopulation2020.csv"
example_country_codes = PGFGLEAM_DIR / "pgfgleam" / "datasets" / "geom_countries_codes.csv"

# Option B: point to your own datasets (uncomment and edit)
# example_flight_matrix = Path("/path/to/your_flight_matrix.csv")
# example_population = Path("/path/to/your_population.csv")
# example_country_codes = Path("/path/to/your_country_codes.csv")

print("Flight matrix:", example_flight_matrix)
print("Population:", example_population)
print("Country codes:", example_country_codes)

setup = DataSetup(
    flight_matrix_path=str(example_flight_matrix),
    population_path=str(example_population),
    country_codes_path=str(example_country_codes),
)

mobility = setup.load_and_process_data()

print("Keys in mobility dict:", list(mobility.keys()))


## 2. Specify high-level epidemiological settings

The ICU + wastewater simulations in `NBPMscape` use parameter combinations like:

- Basic reproduction number $R_0$
- Mean generation time (days)
- Observation window and ICU sampling proportion

These are ultimately controlled inside the Julia script `NBPMscape/AWW_ICU_examples/full_ICU_WW.jl`, but we collect them here so you have a single place to document and tweak your choices.

> **Tip:** For publication-quality work, you should also read and adapt `NBPMscape/parameters.md`, which documents the full set of parameters and literature sources.


In [ ]:
# Example epidemiological settings (edit as needed)

R0_values = [2.0, 2.5, 3.0]
mean_generation_times = [4.0, 6.0]  # in days
icu_sampling_proportion = 0.10      # fraction of ICU cases sampled
max_detection_time_threshold = 200.0
extra_time = 45.0                   # extra days beyond mean detection time

settings = {
    "R0_values": R0_values,
    "mean_generation_times": mean_generation_times,
    "icu_sampling_proportion": icu_sampling_proportion,
    "max_detection_time_threshold": max_detection_time_threshold,
    "extra_time": extra_time,
}

settings


## 3. Prepare or inspect `daily_imports_sensitivity.csv`

The Julia script `full_ICU_WW.jl` expects an input CSV with at least the following columns:

- `R0`
- `generation_time`
- `outbreak_country`
- `mean_detection_time`
- `time`
- `daily_transmission_capable_imports`
- `daily_detectable_imports`

An example file, `daily_imports_sensitivity.csv`, is already provided in `pgfgleam/`. You can:

- Use it as-is to reproduce the existing analysis, or
- Use it as a **template**: create your own version with t=he same columns and save it to `pgfgleam/daily_imports_sensitivity.csv`.

The cell below shows how to load and quickly inspect this file.


In [ ]:
if DEFAULT_DAILY_IMPORTS.exists():
    daily_imports = pd.read_csv(DEFAULT_DAILY_IMPORTS)
    display(daily_imports.head())
    print("\nColumns:", list(daily_imports.columns))
    print("Number of rows:", len(daily_imports))
else:
    print("No daily_imports_sensitivity.csv found at", DEFAULT_DAILY_IMPORTS)
    print("Create one (using this notebook or your own code) before running the Julia simulations.")


## 4. (Optional) Export your own daily imports file

If you derive your own daily imports trajectories using the `mobility` object above and additional modelling code, you should
ultimately produce a `pandas.DataFrame` with the same columns as `daily_imports_sensitivity.csv`.

You can then save it to the location expected by `full_ICU_WW.jl`:

```python
my_daily_imports.to_csv(DEFAULT_DAILY_IMPORTS, index=False)
```

Below we provide a placeholder cell where you can plug in your own logic. By default it simply re-saves the existing
example file, which is useful for testing the full pipeline.


In [ ]:
# TODO: replace this with code that builds `my_daily_imports` from `mobility`
# and your chosen epidemiological assumptions.

if DEFAULT_DAILY_IMPORTS.exists():
    daily_imports = pd.read_csv(DEFAULT_DAILY_IMPORTS)
    print("Using existing daily_imports_sensitivity.csv as template.")
    daily_imports.to_csv(DEFAULT_DAILY_IMPORTS, index=False)
    print("Re-saved to", DEFAULT_DAILY_IMPORTS)
else:
    print("No daily_imports_sensitivity.csv found to use as a template.")
    print("Create a DataFrame named `my_daily_imports` with the required columns and then run:")
    print("    my_daily_imports.to_csv(DEFAULT_DAILY_IMPORTS, index=False)")


## 5. Run the ICU + wastewater simulations (Julia)

Once `daily_imports_sensitivity.csv` exists at the expected location, you can run the ICU + wastewater simulations defined in
`NBPMscape/AWW_ICU_examples/full_ICU_WW.jl`.

The recommended way is from the shell (see the README), but for convenience you can also trigger Julia from this notebook using
`subprocess` as shown below.

> **Note:** This assumes that `julia` is on your `PATH` and that you have already instantiated the NBPMscape environment.


In [ ]:
import subprocess

julia_project = NBPMSCAPE_DIR
julia_script = NBPMSCAPE_DIR / "AWW_ICU_examples" / "full_ICU_WW.jl"

print("Julia project:", julia_project)
print("Julia script:", julia_script)

run_from_notebook = False  # set to True if you want to launch Julia from here

if run_from_notebook:
    cmd = [
        "julia",
        f"--project={julia_project}",
        str(julia_script),
    ]
    print("Running:", " ".join(map(str, cmd)))
    completed = subprocess.run(cmd, check=True)
    print("Julia exit code:", completed.returncode)
else:
    print("To run from a shell, for example:")
    print(f"  cd {julia_project}")
    print(f"  julia --project=. AWW_ICU_examples/full_ICU_WW.jl")
